In [2]:
from pulser import PulseGenerator
import typing as t
import numpy as np

_T = t.TypeVar('_T')

def convert_type(arg: t.Any, converter: _T) -> _T:
    return converter(arg)



In [3]:
readout_time = 10000 # ns 
trig_time = 10 #ns 
initial_delay = 100
singlet_decay = 500
laser_time = 15000 # 15 us 
trig_spot = 70 # where this starts 
clock_time = 10 
MW_buffer_time = 100
awg_trig_time = 10

def Rabi(params):
    '''
    Rabi sequence
    '''
    ## Run a MW pulse of varying duration, then measure the signal
    ## and reference counts from NV.
    # self.total_time = 0
    longest_time = convert_type(round(max(params)), float)
    ## we can measure the pi time on x and on y.
    ## they should be the same, but they technically
    ## have different offsets on our pulse streamer.

    def SingleRabi(iq_on):
        '''
        CREATE SINGLE RABI SEQUENCE TO REPEAT THROUGHOUT EXPERIMENT
        '''

        iq_on = float(round(iq_on)) # convert to proper data type to avoid undesired rpyc netref data type

        '''
        DEFINE SPECIAL TIME INTERVALS FOR EXPERIMENT
        '''
        # padding time to equalize duration of every run (for different vsg_on durations)
        # pad_time = 50000 - self.initial_delay - self.laser_time - self.singlet_decay - iq_on - self.MW_buffer_time - self.readout_time 
        pad_time = longest_time - iq_on

        '''
        DEFINE RELEVANT ON, OFF TIMES FOR DEVICES
        '''

        laser_off1 = initial_delay 
        laser_off2 = singlet_decay + iq_on + MW_buffer_time
        laser_off3 = 100 + pad_time
        # laser_off3 = pad_time + self.rest_time_btw_seqs
        # laser_off4 = laser_off2
        # laser_off5 = self.rest_time_btw_seqs

        # mw I & Q off windows
        iq_off1 = laser_off1 + laser_time + singlet_decay
        iq_off2 = (iq_on - awg_trig_time) + MW_buffer_time + readout_time + laser_off3 # + self.laser_time # + laser_off4 + laser_off5

        # Digitizer trigger timing
        clock_off1 = laser_off1 + laser_time + laser_off2 + trig_spot - clock_time
        clock_off2 = - trig_spot + readout_time + laser_off3
                
        '''
        CONSTRUCT PULSE SEQUENCE
        '''


        # define sequence structure for laser            
        laser_seq = [(laser_off1, 0), (laser_time, 1), (laser_off2, 0), (readout_time, 1), (laser_off3, 0)]
                    #  (laser_off3, 0), (self.laser_time, 1), (laser_off4, 0), (self.readout_time, 1), (laser_off5, 0)]
    
        # define sequence structure for DAQ trigger
        dig_clock_seq = [(clock_off1, 0), (clock_time, 1), (clock_off2, 0)]

        # define sequence structure for MW I and Q when MW = ON
        mw_iq_on_seq = [(iq_off1, 0), (awg_trig_time, 1), (iq_off2, 0)]
        mw_iq_off_seq = [(iq_off1, 0), (awg_trig_time, 0), (iq_off2, 0)]

        # assign sequences to respective channels for seq_on
        laser_ps = laser_seq+laser_seq
        dig_ps = dig_clock_seq+dig_clock_seq
        mw_ps = mw_iq_on_seq+mw_iq_off_seq


        return [laser_ps, dig_ps, mw_ps]

    # seqs = Pulser.createSequence()
    laser_ps_tot=[]
    dig_ps_tot=[]
    mw_ps_tot=[]
    for mw_time in params:
        # seqs += SingleRabi(mw_time)
        laser_ps_tot += SingleRabi(mw_time)[0]
        dig_ps_tot += SingleRabi(mw_time)[1]
        mw_ps_tot += SingleRabi(mw_time)[2]

    return [laser_ps_tot, dig_ps_tot, mw_ps_tot]


In [16]:
# Pulse Streamer 
# PSch_MW=2
# PSch_Digitizer=1 
# PSch_Laser= 3
# PS_chmap={'mw':PSch_MW,
#           'digtrig':PSch_Digitizer,
#           'laser':PSch_Laser}

PSch_MW = 2
PSch_Digitizer = 1 
PSch_Laser = 3
PSch_RF = 7
PS_chmap={'mw':PSch_MW,
          'digtrig':PSch_Digitizer,
          'laser':PSch_Laser,
          'rf': PSch_RF}

ip = '10.135.70.193' 
ps = PulseGenerator(ip,PS_chmap)
INF = np.iinfo(np.int64).max

Connect to Pulse Streamer via JSON-RPC.
IP / Hostname: 10.135.70.193
Pulse Streamer 8/2 firmware: v1.5.2
Client software: v1.7.0
Your client software is more up to date than the Pulse Streamer 8/2 firmware. We recommend updating the firmware of your Pulse Streamer 8/2.
For detailed information visit https://www.swabianinstruments.com/pulse-streamer-8-2/downloads/ or contact support@swabianinstruments.com


In [21]:
def CASR(nuclear_pihalf, laser_time, singlet_decay, pihalf_x, pihalf_y, pi_x, pi_y, tau, n, mw_buffer_time, read_time, wait_time, n_R):
        '''
        Coherent averaged synchronized readout (CASR).
        '''
        nuclear_pihalf = convert_type(round(nuclear_pihalf), float)
        laser_time = convert_type(round(laser_time), float)
        singlet_decay = convert_type(round(singlet_decay), float)
        pihalf_x = convert_type(round(pihalf_x), float)
        pihalf_y = convert_type(round(pihalf_y), float)
        pi_x = convert_type(round(pi_x), float)
        pi_y = convert_type(round(pi_y), float)
        tau = convert_type(round(tau), float)
        n = convert_type(round(n), int)
        mw_buffer_time = convert_type(round(mw_buffer_time), float)
        read_time = convert_type(round(read_time), float)
        wait_time = convert_type(round(wait_time), float)
        n_R = convert_type(round(n_R), int) # number of subsequences (synchronized readouts) per "run"

        def PiPulsesN(tau, N):
            xy8_iq_seq = [(tau/2, 0), (awg_trig_time, 1), 
                          ((pi_x - awg_trig_time) + tau, 0), (awg_trig_time, 1), 
                          ((pi_y - awg_trig_time) + tau, 0), (awg_trig_time, 1), 
                          ((pi_x - awg_trig_time) + tau, 0), (awg_trig_time, 1), 
                          ((pi_y - awg_trig_time) + tau, 0), (awg_trig_time, 1),
                          ((pi_y - awg_trig_time) + tau, 0), (awg_trig_time, 1),
                          ((pi_x - awg_trig_time) + tau, 0), (awg_trig_time, 1),
                          ((pi_y - awg_trig_time) + tau, 0), (awg_trig_time, 1),
                          ((pi_x - awg_trig_time) + tau/2, 0)]

            mw_IQ = (xy8_iq_seq)*N
                
            return mw_IQ

        def SingleCASR():
            # create sequence objects for MW on and off blocks
            # seq_rf = self.Pulser.createSequence()
            # seq = self.Pulser.createSequence()

            # total time for CASR DD subsequence
            casr_time = pihalf_x + (tau/2 + 4*pi_x + 4*pi_y + 7*tau + tau/2)*n + pihalf_y

            # laser       
            laser_off1 = singlet_decay + casr_time + mw_buffer_time
            laser_off2 = wait_time
            laser_seq1 = [(nuclear_pihalf, 0), (laser_time, 1), (laser_off1, 0), (read_time, 1), (laser_off2, 0)]
            laser_seq2 = [(laser_time, 1), (laser_off1, 0), (read_time, 1), (laser_off2, 0)] # define sequence structure for laser

            # digitizer 
            clock_off1 = laser_time + laser_off1
            clock_off2 = - clock_time + read_time + laser_off2
            dig_clock_seq1 = [(nuclear_pihalf, 0), (clock_off1, 0), (clock_time, 1), (clock_off2, 0)] # define sequence structure for digitizer trigger
            dig_clock_seq2 = [(clock_off1, 0), (clock_time, 1), (clock_off2, 0)] # define sequence structure for digitizer trigger

            # mw I & Q off windows 
            iq_off_start = laser_time + singlet_decay
            iq_off_end = (pihalf_y - awg_trig_time) + mw_buffer_time + read_time + laser_off2
            mw_iq_seq1 = [(nuclear_pihalf, 0), (iq_off_start, 0), (awg_trig_time, 1), (pihalf_x - awg_trig_time, 0)] + PiPulsesN(tau, n) + [(awg_trig_time, 1), (iq_off_end, 0)] # sequence structure for I & Q MW channels             
            mw_iq_seq2 = [(iq_off_start, 0), (awg_trig_time, 1), (pihalf_x - awg_trig_time, 0)] + PiPulsesN(tau, n) + [(awg_trig_time, 1), (iq_off_end, 0)] # sequence structure for I & Q MW channels 

            # nuclear spin pi/2 initial pulse
            nuclear_spin_off = - awg_trig_time + nuclear_pihalf
            nuclear_spin_seq = [(awg_trig_time, 1), (nuclear_spin_off, 0)]
            
            # assign sequences to respective channels for seq_on
            rf_ps = nuclear_spin_seq
            laser_ps = laser_seq1 + laser_seq2*(n_R-1)
            dig_ps = dig_clock_seq1 + dig_clock_seq2*(n_R-1)
            mw_ps = mw_iq_seq1 + mw_iq_seq2*(n_R-1)

            return [rf_ps, laser_ps, dig_ps, mw_ps]

        # seqs = Pulser.createSequence()
        rf_ps_tot = SingleCASR()[0]
        laser_ps_tot = SingleCASR()[1]
        dig_ps_tot = SingleCASR()[2]
        mw_ps_tot = SingleCASR()[3]

        return [rf_ps_tot, laser_ps_tot, dig_ps_tot, mw_ps_tot]

        #     # assign initial nuclear spin pi/2 pulse to seq_init
        #     seq_rf.setDigital(7, nuclear_spin_seq)

        #     # assign sequences to respective channels for seq
        #     seq.setDigital(3, laser_seq) # laser
        #     seq.setDigital(1, dig_clock_seq) # digitizer trigger
        #     seq.setDigital(2, mw_iq_seq) # MW IQ

        #     return seq_rf + seq*n_R # + seq_rf + seq*n_R
        
        # seqs = SingleCASR()

        # return seqs

In [23]:
# mw_times = np.linspace(200e-9, 200e-9, 1) * 1e9
nuclear_pihalf = 10000 # ns 
pihalf_x = 16
pihalf_y = pihalf_x
pi_x = 32 # ns
pi_y = pi_x

tau = (1/(2*1.3333e6))*1e9
n = 1
wait_time = 300
n_R = 3

readout_time = 2500 # ns 
trig_time = 10 #ns 
singlet_decay = 500
laser_time = 10000 # 15 us 
clock_time = 10 
mw_buffer_time = 100
awg_trig_time = 10

sequence = CASR(nuclear_pihalf, laser_time, singlet_decay, pihalf_x, pihalf_y, pi_x, pi_y, tau, n, mw_buffer_time, readout_time, wait_time, n_R)

ps.setDigital("rf", sequence[0])
ps.setDigital("laser", sequence[1]) # digitizer trigger
ps.setDigital("digtrig", sequence[2]) # digitizer trigger
ps.setDigital("mw", sequence[3]) # MW IQ
ps.setTrigger
ps.plotSeq(plot_all=False)